# Standard ONNX Operators — Hands-On Application

## Objective

Master building ONNX computational graphs using standard operators: CNN blocks, linear layers,
activations, and attention patterns. Verify numerical correctness against NumPy and explore
operator attributes and composition patterns.

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Setup](#1-setup) | Imports and utilities |
| 2 | [Exercise 1: CNN Block Builder](#2-exercise-1) | Conv → BN → ReLU → Pool |
| 3 | [Exercise 2: Linear Layer with Verification](#3-exercise-2) | MatMul + Add vs NumPy |
| 4 | [Exercise 3: Operator Attributes Deep Dive](#4-exercise-3) | Inspect and modify attributes |
| 5 | [Exercise 4: Unique Op Analysis](#5-exercise-4) | Profile operator usage in models |
| 6 | [Exercise 5: Numerical Verification vs NumPy](#6-exercise-5) | Systematic correctness testing |
| 7 | [Exercise 6: Operator Composition Patterns](#7-exercise-6) | Common building blocks |
| 8 | [Exercise 7: Performance Comparison](#8-exercise-7) | Benchmark equivalent op patterns |
| 9 | [Challenge: Mini-VGG Block](#9-challenge) | Multi-layer CNN architecture |
| 10 | [Summary](#10-summary) | Skills review |

In [ ]:
# 1. Setup <a id="1-setup"></a>
# !pip install onnx numpy matplotlib --quiet

import numpy as np
import onnx
from onnx import helper, TensorProto, checker, numpy_helper, defs, shape_inference
from onnx.reference import ReferenceEvaluator
import time
import matplotlib.pyplot as plt


def run_model(model, feeds):
    """Run a model with the ONNX reference evaluator."""
    ev = ReferenceEvaluator(model)
    return ev.run(None, feeds)


def build_and_check(nodes, inputs, outputs, initializers=None, name="graph"):
    """Build, validate, and return an ONNX model."""
    graph = helper.make_graph(
        nodes, name, inputs, outputs, initializer=initializers or []
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    model = shape_inference.infer_shapes(model)
    checker.check_model(model)
    return model


print(f"ONNX version: {onnx.__version__}, OpSet: {defs.onnx_opset_version()}")

## 2. Exercise 1: CNN Block Builder <a id="2-exercise-1"></a>

Build a reusable CNN block: **Conv → BatchNorm → ReLU → MaxPool**.

**Output spatial dimensions for Conv:**
$$H_{out} = \left\lfloor \frac{H_{in} + p_t + p_b - d_h(k_h - 1) - 1}{s_h} \right\rfloor + 1$$

**For MaxPool with kernel $k$ and stride $s$:**
$$H_{out} = \left\lfloor \frac{H_{in} - k}{s} \right\rfloor + 1$$

In [ ]:
def make_cnn_block(prefix: str, c_in: int, c_out: int,
                   input_name: str = "X", output_name: str = "Y",
                   kernel: int = 3, pool_size: int = 2):
    """Create a Conv-BN-ReLU-MaxPool block."""
    pad = kernel // 2  # same padding for conv
    nodes = [
        helper.make_node("Conv", [input_name, f"{prefix}.w", f"{prefix}.b"],
                         [f"{prefix}.conv"], kernel_shape=[kernel, kernel],
                         pads=[pad, pad, pad, pad]),
        helper.make_node("BatchNormalization",
                         [f"{prefix}.conv", f"{prefix}.bn_s", f"{prefix}.bn_b",
                          f"{prefix}.bn_m", f"{prefix}.bn_v"],
                         [f"{prefix}.bn"], epsilon=1e-5),
        helper.make_node("Relu", [f"{prefix}.bn"], [f"{prefix}.relu"]),
        helper.make_node("MaxPool", [f"{prefix}.relu"], [output_name],
                         kernel_shape=[pool_size, pool_size],
                         strides=[pool_size, pool_size]),
    ]
    inits = [
        numpy_helper.from_array(np.random.randn(c_out, c_in, kernel, kernel).astype(np.float32) * 0.1, f"{prefix}.w"),
        numpy_helper.from_array(np.zeros(c_out, dtype=np.float32), f"{prefix}.b"),
        numpy_helper.from_array(np.ones(c_out, dtype=np.float32), f"{prefix}.bn_s"),
        numpy_helper.from_array(np.zeros(c_out, dtype=np.float32), f"{prefix}.bn_b"),
        numpy_helper.from_array(np.zeros(c_out, dtype=np.float32), f"{prefix}.bn_m"),
        numpy_helper.from_array(np.ones(c_out, dtype=np.float32), f"{prefix}.bn_v"),
    ]
    return nodes, inits


# Build a 3-block CNN: 1→16→32→64, spatial 28→14→7→3
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 1, 28, 28])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

nodes1, inits1 = make_cnn_block("b1", 1, 16, "X", "pool1")
nodes2, inits2 = make_cnn_block("b2", 16, 32, "pool1", "pool2")
nodes3, inits3 = make_cnn_block("b3", 32, 64, "pool2", "Y")

cnn_model = build_and_check(
    nodes1 + nodes2 + nodes3, [X], [Y],
    inits1 + inits2 + inits3, "three_block_cnn"
)

# Run and verify shapes
x_test = np.random.randn(1, 1, 28, 28).astype(np.float32)
result = run_model(cnn_model, {"X": x_test})
print(f"CNN architecture: {len(cnn_model.graph.node)} ops")
print(f"Ops used: {sorted(set(n.op_type for n in cnn_model.graph.node))}")
print(f"Input:  {x_test.shape}")
print(f"Output: {result[0].shape}")

# Verify expected output shape: 28/2/2/2 = 3 (with pad)
# Actually 28→14→7→3 (pool2 with floor)
assert result[0].shape[0] == 1   # batch
assert result[0].shape[1] == 64  # channels
print(f"\nSpatial reduction: 28 → {result[0].shape[2]} (factor: {28/result[0].shape[2]:.1f}x)")
print("CNN block builder verified. ✓")

## 3. Exercise 2: Linear Layer with Verification <a id="3-exercise-2"></a>

Build a linear transformation: $Y = XW + b$ and verify against NumPy.

**Verification criterion:**
$$\|Y_{\text{ONNX}} - Y_{\text{NumPy}}\|_\infty < \epsilon, \quad \epsilon = 10^{-5}$$

In [ ]:
in_dim, out_dim = 256, 128
np.random.seed(42)

W_data = np.random.randn(in_dim, out_dim).astype(np.float32) * 0.01
b_data = np.random.randn(out_dim).astype(np.float32) * 0.01

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", in_dim])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", out_dim])
W = numpy_helper.from_array(W_data, "W")
b = numpy_helper.from_array(b_data, "b")

# Method 1: MatMul + Add
linear_v1 = build_and_check(
    [helper.make_node("MatMul", ["X", "W"], ["mm"]),
     helper.make_node("Add", ["mm", "b"], ["Y"])],
    [X], [Y], [W, b], "linear_matmul_add"
)

# Method 2: Gemm (fused)
W_gemm = numpy_helper.from_array(W_data.T, "W_gemm")  # Gemm expects transposed
linear_v2 = build_and_check(
    [helper.make_node("Gemm", ["X", "W_gemm", "b"], ["Y"], transB=1)],
    [X], [Y], [W_gemm, b], "linear_gemm"
)

# Run both and compare with NumPy
batch_sizes = [1, 4, 16, 64]

print(f"Linear layer: {in_dim} → {out_dim}")
print(f"{'Batch':<6} | {'NumPy':>10} | {'MatMul+Add':>12} | {'Gemm':>10} | {'Max Diff':>10}")
print("-" * 60)

for bs in batch_sizes:
    x_test = np.random.randn(bs, in_dim).astype(np.float32)

    # NumPy reference
    y_numpy = x_test @ W_data + b_data

    # ONNX methods
    y_v1 = run_model(linear_v1, {"X": x_test})[0]
    y_v2 = run_model(linear_v2, {"X": x_test})[0]

    diff_v1 = np.abs(y_v1 - y_numpy).max()
    diff_v2 = np.abs(y_v2 - y_numpy).max()
    diff_12 = np.abs(y_v1 - y_v2).max()

    print(f"{bs:<6} | {y_numpy.mean():>10.6f} | err={diff_v1:>8.2e} | err={diff_v2:>8.2e} | v1-v2={diff_12:.2e}")

    assert diff_v1 < 1e-5, f"MatMul+Add error too large: {diff_v1}"
    assert diff_v2 < 1e-5, f"Gemm error too large: {diff_v2}"

print("\nBoth linear implementations match NumPy within epsilon. ✓")

## 4. Exercise 3: Operator Attributes Deep Dive <a id="4-exercise-3"></a>

Explore how operator attributes control behavior. Each ONNX operator has:
- **Required attributes**: must be specified (e.g., Conv's `kernel_shape`)
- **Optional attributes**: have defaults (e.g., Conv's `dilations=[1,1]`)

Attributes modify the operation's semantics without changing the graph structure.

In [ ]:
from onnx import AttributeProto

def inspect_node_attributes(node):
    """Detailed attribute inspection for a node."""
    type_names = {1: 'FLOAT', 2: 'INT', 3: 'STRING', 4: 'TENSOR',
                  5: 'GRAPH', 6: 'FLOATS', 7: 'INTS', 8: 'STRINGS'}
    print(f"  {node.op_type} ({len(node.attribute)} attributes):")
    for attr in node.attribute:
        t = type_names.get(attr.type, f"TYPE_{attr.type}")
        if attr.type == AttributeProto.INT:
            val = attr.i
        elif attr.type == AttributeProto.INTS:
            val = list(attr.ints)
        elif attr.type == AttributeProto.FLOAT:
            val = attr.f
        elif attr.type == AttributeProto.FLOATS:
            val = list(attr.floats)
        elif attr.type == AttributeProto.STRING:
            val = attr.s.decode()
        else:
            val = "<complex>"
        print(f"    {attr.name:<20} {t:<8} = {val}")


# Create nodes with various attribute configurations
test_nodes = [
    helper.make_node("Conv", ["X", "W"], ["Y"],
                     kernel_shape=[5, 5], strides=[2, 2],
                     pads=[2, 2, 2, 2], dilations=[1, 1], group=1),
    helper.make_node("MaxPool", ["X"], ["Y"],
                     kernel_shape=[3, 3], strides=[2, 2], pads=[1, 1, 1, 1]),
    helper.make_node("BatchNormalization", ["X", "s", "b", "m", "v"], ["Y"],
                     epsilon=1e-5, momentum=0.9),
    helper.make_node("Gemm", ["A", "B", "C"], ["Y"],
                     alpha=1.0, beta=1.0, transA=0, transB=1),
    helper.make_node("Softmax", ["X"], ["Y"], axis=-1),
    helper.make_node("Conv", ["X", "W"], ["Y"],
                     kernel_shape=[3, 3], dilations=[2, 2],
                     pads=[2, 2, 2, 2], group=32),  # depthwise-like
]

for node in test_nodes:
    inspect_node_attributes(node)
    print()

# Demonstrate attribute effect: dilated conv
print("Effect of dilation on receptive field:")
for dilation in [1, 2, 4]:
    effective_k = 3 + (3 - 1) * (dilation - 1)
    print(f"  kernel=3, dilation={dilation} → effective receptive field = {effective_k}x{effective_k}")

## 5. Exercise 4: Unique Op Analysis <a id="5-exercise-4"></a>

Build a tool that profiles the unique operators used in a model, counting occurrences
and categorizing them by function.

In [ ]:
OP_CATEGORIES = {
    "Conv": "convolution", "ConvTranspose": "convolution",
    "MatMul": "linear", "Gemm": "linear",
    "Relu": "activation", "Sigmoid": "activation", "Tanh": "activation",
    "LeakyRelu": "activation", "Softmax": "activation",
    "BatchNormalization": "normalization", "LayerNormalization": "normalization",
    "MaxPool": "pooling", "AveragePool": "pooling", "GlobalAveragePool": "pooling",
    "Add": "arithmetic", "Sub": "arithmetic", "Mul": "arithmetic", "Div": "arithmetic",
    "Reshape": "shape", "Transpose": "shape", "Flatten": "shape",
    "Concat": "data_movement", "Split": "data_movement", "Gather": "data_movement",
}


def analyze_operators(model: onnx.ModelProto, label: str = "model") -> dict:
    """Analyze operator usage patterns."""
    op_counts = {}
    for node in model.graph.node:
        op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1

    # Categorize
    cat_counts = {}
    for op, count in op_counts.items():
        cat = OP_CATEGORIES.get(op, "other")
        cat_counts[cat] = cat_counts.get(cat, 0) + count

    print(f"\n{'═'*50}")
    print(f"  Operator Analysis: {label}")
    print(f"{'═'*50}")
    print(f"  Total nodes: {sum(op_counts.values())}")
    print(f"  Unique ops:  {len(op_counts)}")
    print(f"\n  {'Op Type':<22} {'Count':>5} {'Category':<15} Bar")
    print(f"  {'-'*55}")
    for op, count in sorted(op_counts.items(), key=lambda x: -x[1]):
        cat = OP_CATEGORIES.get(op, "other")
        bar = "█" * count
        print(f"  {op:<22} {count:>5} {cat:<15} {bar}")

    return {"op_counts": op_counts, "cat_counts": cat_counts}


# Analyze the CNN model
cnn_analysis = analyze_operators(cnn_model, "3-Block CNN")

# Build a simple transformer-like model for comparison
d = 64
Xt = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 8, d])
Yt = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, 8, d])
t_inits = [
    numpy_helper.from_array(np.random.randn(d, d).astype(np.float32) * 0.01, "Wq"),
    numpy_helper.from_array(np.random.randn(d, d).astype(np.float32) * 0.01, "Wk"),
    numpy_helper.from_array(np.random.randn(d, d).astype(np.float32) * 0.01, "Wv"),
    numpy_helper.from_array(np.array([d**0.5], dtype=np.float32), "scale"),
]
t_nodes = [
    helper.make_node("MatMul", ["X", "Wq"], ["Q"]),
    helper.make_node("MatMul", ["X", "Wk"], ["K"]),
    helper.make_node("MatMul", ["X", "Wv"], ["V"]),
    helper.make_node("Transpose", ["K"], ["Kt"], perm=[0, 2, 1]),
    helper.make_node("MatMul", ["Q", "Kt"], ["scores"]),
    helper.make_node("Div", ["scores", "scale"], ["scaled"]),
    helper.make_node("Softmax", ["scaled"], ["attn"], axis=-1),
    helper.make_node("MatMul", ["attn", "V"], ["context"]),
    helper.make_node("Add", ["context", "X"], ["Y"]),
]
attn_model = build_and_check(t_nodes, [Xt], [Yt], t_inits, "attention")
attn_analysis = analyze_operators(attn_model, "Self-Attention")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (analysis, title) in zip(axes, [(cnn_analysis, "CNN"), (attn_analysis, "Attention")]):
    cats = analysis["cat_counts"]
    ax.pie(cats.values(), labels=cats.keys(), autopct="%1.0f%%", startangle=90)
    ax.set_title(f"{title}: Op Category Distribution", fontweight="bold")

plt.tight_layout()
plt.show()

## 6. Exercise 5: Numerical Verification vs NumPy <a id="6-exercise-5"></a>

Systematically verify ONNX operators against NumPy reference implementations.
This builds confidence that the ONNX computation graph computes what we expect.

For each operator $f$:
$$\text{error} = \|f_{\text{ONNX}}(\mathbf{x}) - f_{\text{NumPy}}(\mathbf{x})\|_\infty$$

In [ ]:
def verify_op(op_type, inputs_spec, numpy_fn, attrs=None, description=""):
    """Verify an ONNX op against a NumPy reference."""
    attrs = attrs or {}

    # Build ONNX model
    input_infos = []
    feed_dict = {}
    input_names = []

    for i, (name, data) in enumerate(inputs_spec):
        input_infos.append(
            helper.make_tensor_value_info(name, TensorProto.FLOAT, list(data.shape))
        )
        feed_dict[name] = data
        input_names.append(name)

    node = helper.make_node(op_type, input_names, ["Y"], **attrs)
    output = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
    graph = helper.make_graph([node], "verify", input_infos, [output])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

    # Run ONNX
    y_onnx = run_model(model, feed_dict)[0]

    # Run NumPy
    y_numpy = numpy_fn(*[data for _, data in inputs_spec])

    # Compare
    max_err = np.abs(y_onnx - y_numpy).max()
    mean_err = np.abs(y_onnx - y_numpy).mean()
    match = max_err < 1e-5

    return {"op": op_type, "desc": description, "match": match,
            "max_err": max_err, "mean_err": mean_err,
            "shape_onnx": y_onnx.shape, "shape_numpy": y_numpy.shape}


np.random.seed(42)
x = np.random.randn(4, 64).astype(np.float32)
a = np.random.randn(4, 64).astype(np.float32)
b_vec = np.random.randn(4, 64).astype(np.float32)

# Verify multiple operators
tests = [
    ("Add", [("X", x), ("Y", b_vec)], lambda x, y: x + y, {}, "element-wise add"),
    ("Sub", [("X", x), ("Y", b_vec)], lambda x, y: x - y, {}, "element-wise sub"),
    ("Mul", [("X", x), ("Y", b_vec)], lambda x, y: x * y, {}, "element-wise mul"),
    ("Div", [("X", x), ("Y", np.abs(b_vec) + 0.01)], lambda x, y: x / y, {}, "element-wise div"),
    ("Relu", [("X", x)], lambda x: np.maximum(x, 0), {}, "ReLU activation"),
    ("Sigmoid", [("X", x)], lambda x: 1 / (1 + np.exp(-x)), {}, "Sigmoid activation"),
    ("Tanh", [("X", x)], lambda x: np.tanh(x), {}, "Tanh activation"),
    ("Exp", [("X", x * 0.1)], lambda x: np.exp(x), {}, "Exponential"),
    ("Sqrt", [("X", np.abs(x) + 0.01)], lambda x: np.sqrt(x), {}, "Square root"),
]

print(f"{'Op':<12} | {'Description':<20} | {'Match':>5} | {'Max Error':>10} | {'Mean Error':>10}")
print("-" * 70)

all_pass = True
for op, inputs, fn, attrs, desc in tests:
    r = verify_op(op, inputs, fn, attrs, desc)
    status = "✓" if r["match"] else "✗"
    print(f"{r['op']:<12} | {r['desc']:<20} | {status:>5} | {r['max_err']:>10.2e} | {r['mean_err']:>10.2e}")
    if not r["match"]:
        all_pass = False

assert all_pass, "Some operators failed verification!"
print(f"\nAll {len(tests)} operators verified against NumPy. ✓")

## 7. Exercise 6: Operator Composition Patterns <a id="7-exercise-6"></a>

Common neural network building blocks composed from ONNX operators:

| Pattern | Ops | Formula |
|---------|-----|--------|
| Linear | MatMul + Add | $Y = XW + b$ |
| LayerNorm | ReduceMean + Sub + Mul + Sqrt + Div + Mul + Add | $\frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$ |
| GELU | Mul + Div + Erf + Add + Mul | $x \cdot \Phi(x)$ |
| Residual | Add | $Y = F(X) + X$ |

In [ ]:
def build_gelu_model(shape):
    """GELU: x * 0.5 * (1 + erf(x / sqrt(2)))"""
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, list(shape))
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, list(shape))

    sqrt2 = numpy_helper.from_array(np.array([np.sqrt(2.0)], dtype=np.float32), "sqrt2")
    one = numpy_helper.from_array(np.array([1.0], dtype=np.float32), "one")
    half = numpy_helper.from_array(np.array([0.5], dtype=np.float32), "half")

    nodes = [
        helper.make_node("Div", ["X", "sqrt2"], ["x_norm"]),
        helper.make_node("Erf", ["x_norm"], ["erf_out"]),
        helper.make_node("Add", ["erf_out", "one"], ["erf_plus1"]),
        helper.make_node("Mul", ["X", "erf_plus1"], ["x_times"]),
        helper.make_node("Mul", ["x_times", "half"], ["Y"]),
    ]

    return build_and_check(nodes, [X], [Y], [sqrt2, one, half], "gelu")


def build_swish_model(shape):
    """Swish: x * sigmoid(x)"""
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, list(shape))
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, list(shape))

    nodes = [
        helper.make_node("Sigmoid", ["X"], ["sig"]),
        helper.make_node("Mul", ["X", "sig"], ["Y"]),
    ]

    return build_and_check(nodes, [X], [Y], [], "swish")


# Build and compare activation patterns
shape = [1, 200]
x_test = np.linspace(-4, 4, 200).reshape(1, 200).astype(np.float32)

gelu_model = build_gelu_model(shape)
swish_model = build_swish_model(shape)

y_gelu = run_model(gelu_model, {"X": x_test})[0]
y_swish = run_model(swish_model, {"X": x_test})[0]

# NumPy references
from scipy.special import erf as scipy_erf
try:
    y_gelu_np = x_test * 0.5 * (1 + scipy_erf(x_test / np.sqrt(2)))
    gelu_err = np.abs(y_gelu - y_gelu_np).max()
    print(f"GELU max error vs SciPy: {gelu_err:.2e}")
except ImportError:
    print("SciPy not available, skipping GELU verification")

y_swish_np = x_test * (1 / (1 + np.exp(-x_test)))
swish_err = np.abs(y_swish - y_swish_np).max()
print(f"Swish max error vs NumPy: {swish_err:.2e}")
assert swish_err < 1e-5

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_test[0], y_gelu[0], label="GELU (5 ops)", linewidth=2)
ax.plot(x_test[0], y_swish[0], label="Swish (2 ops)", linewidth=2)
ax.plot(x_test[0], np.maximum(x_test[0], 0), "--", label="ReLU (1 op)", alpha=0.7)
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.set_xlabel("Input"); ax.set_ylabel("Output")
ax.set_title("Composite Activation Functions from ONNX Ops", fontweight="bold")
ax.legend(fontsize=11); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f"\nGELU: {len(gelu_model.graph.node)} ops, Swish: {len(swish_model.graph.node)} ops")

## 8. Exercise 7: Performance Comparison <a id="8-exercise-7"></a>

Compare execution time of equivalent operator patterns. ONNX provides multiple ways
to express the same computation; fused operators are often faster.

Test: $Y = \alpha \cdot AB + \beta \cdot C$ via:
1. `MatMul + Mul + MatMul + Mul + Add` (decomposed)
2. `Gemm` (fused)

In [ ]:
M, K, N = 64, 128, 32
np.random.seed(0)

A_data = np.random.randn(M, K).astype(np.float32)
B_data = np.random.randn(K, N).astype(np.float32)
C_data = np.random.randn(M, N).astype(np.float32)

# Method 1: Decomposed
A_in = helper.make_tensor_value_info("A", TensorProto.FLOAT, [M, K])
Y_out = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [M, N])
B_init = numpy_helper.from_array(B_data, "B")
C_init = numpy_helper.from_array(C_data, "C")

decomposed = build_and_check(
    [
        helper.make_node("MatMul", ["A", "B"], ["AB"]),
        helper.make_node("Add", ["AB", "C"], ["Y"]),
    ],
    [A_in], [Y_out], [B_init, C_init], "decomposed"
)

# Method 2: Gemm (fused)
B_T = numpy_helper.from_array(B_data.T, "B_T")
fused = build_and_check(
    [helper.make_node("Gemm", ["A", "B_T", "C"], ["Y"], transB=1)],
    [A_in], [Y_out], [B_T, C_init], "fused_gemm"
)

# Benchmark
n_runs = 100
feed = {"A": A_data}

# Warmup
run_model(decomposed, feed)
run_model(fused, feed)

t0 = time.perf_counter()
for _ in range(n_runs):
    y_dec = run_model(decomposed, feed)[0]
t_decomposed = (time.perf_counter() - t0) / n_runs * 1000

t0 = time.perf_counter()
for _ in range(n_runs):
    y_fused = run_model(fused, feed)[0]
t_fused = (time.perf_counter() - t0) / n_runs * 1000

# Verify equivalence
max_diff = np.abs(y_dec - y_fused).max()
assert max_diff < 1e-5, f"Results differ: {max_diff}"

print(f"Performance Comparison ({M}x{K} @ {K}x{N}, {n_runs} runs):")
print(f"  Decomposed (MatMul+Add): {t_decomposed:.3f} ms/run")
print(f"  Fused (Gemm):            {t_fused:.3f} ms/run")
print(f"  Speedup:                 {t_decomposed/t_fused:.2f}x")
print(f"  Max diff:                {max_diff:.2e}")
print(f"  Results equivalent:      ✓")

## 9. Challenge: Mini-VGG Block <a id="9-challenge"></a>

Build a VGG-style network block:
```
Input [N,1,32,32]
 → Conv3x3(64) → ReLU → Conv3x3(64) → ReLU → MaxPool2x2
 → Conv3x3(128) → ReLU → Conv3x3(128) → ReLU → MaxPool2x2
 → Flatten → Linear(2048→256) → ReLU → Linear(256→10)
Output [N,10]
```

Verify shapes through the network and count total parameters.

In [ ]:
np.random.seed(123)

def conv_relu(prefix, c_in, c_out, inp, out):
    """Conv3x3 + ReLU."""
    nodes = [
        helper.make_node("Conv", [inp, f"{prefix}.w", f"{prefix}.b"], [f"{prefix}.c"],
                         kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node("Relu", [f"{prefix}.c"], [out]),
    ]
    inits = [
        numpy_helper.from_array(np.random.randn(c_out, c_in, 3, 3).astype(np.float32) * 0.05, f"{prefix}.w"),
        numpy_helper.from_array(np.zeros(c_out, dtype=np.float32), f"{prefix}.b"),
    ]
    return nodes, inits


# Block 1: 1→64, spatial 32→16
n1, i1 = conv_relu("c1", 1, 64, "X", "r1")
n2, i2 = conv_relu("c2", 64, 64, "r1", "r2")
pool1 = [helper.make_node("MaxPool", ["r2"], ["p1"], kernel_shape=[2, 2], strides=[2, 2])]

# Block 2: 64→128, spatial 16→8
n3, i3 = conv_relu("c3", 64, 128, "p1", "r3")
n4, i4 = conv_relu("c4", 128, 128, "r3", "r4")
pool2 = [helper.make_node("MaxPool", ["r4"], ["p2"], kernel_shape=[2, 2], strides=[2, 2])]

# Classifier head: Flatten → Linear(128*8*8→256) → ReLU → Linear(256→10)
flatten_shape = numpy_helper.from_array(np.array([0, -1], dtype=np.int64), "flat_shape")
fc1_w = numpy_helper.from_array(np.random.randn(128 * 8 * 8, 256).astype(np.float32) * 0.01, "fc1.w")
fc1_b = numpy_helper.from_array(np.zeros(256, dtype=np.float32), "fc1.b")
fc2_w = numpy_helper.from_array(np.random.randn(256, 10).astype(np.float32) * 0.01, "fc2.w")
fc2_b = numpy_helper.from_array(np.zeros(10, dtype=np.float32), "fc2.b")

head_nodes = [
    helper.make_node("Reshape", ["p2", "flat_shape"], ["flat"]),
    helper.make_node("MatMul", ["flat", "fc1.w"], ["fc1_mm"]),
    helper.make_node("Add", ["fc1_mm", "fc1.b"], ["fc1_out"]),
    helper.make_node("Relu", ["fc1_out"], ["fc1_relu"]),
    helper.make_node("MatMul", ["fc1_relu", "fc2.w"], ["fc2_mm"]),
    helper.make_node("Add", ["fc2_mm", "fc2.b"], ["Y"]),
]

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 1, 32, 32])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

all_nodes = n1 + n2 + pool1 + n3 + n4 + pool2 + head_nodes
all_inits = i1 + i2 + i3 + i4 + [flatten_shape, fc1_w, fc1_b, fc2_w, fc2_b]

vgg_model = build_and_check(all_nodes, [X], [Y], all_inits, "mini_vgg")

# Run inference
x_test = np.random.randn(1, 1, 32, 32).astype(np.float32)
result = run_model(vgg_model, {"X": x_test})

# Parameter count
total_params = sum(int(np.prod(list(i.dims))) for i in vgg_model.graph.initializer
                   if i.data_type == TensorProto.FLOAT)

print("Mini-VGG Architecture:")
print(f"  Input:      {x_test.shape}")
print(f"  Output:     {result[0].shape}")
print(f"  Total ops:  {len(vgg_model.graph.node)}")
print(f"  Parameters: {total_params:,}")
print(f"  Op types:   {sorted(set(n.op_type for n in vgg_model.graph.node))}")
print(f"  Model size: {len(vgg_model.SerializeToString()):,} bytes")

# Shape inference verification
assert result[0].shape == (1, 10), f"Expected (1,10), got {result[0].shape}"
print(f"\nOutput shape (1, 10) verified. ✓")

# Layer-by-layer shapes
print(f"\nShape progression:")
print(f"  Input:      [1, 1, 32, 32]")
print(f"  After B1:   [1, 64, 16, 16]  (pool /2)")
print(f"  After B2:   [1, 128, 8, 8]   (pool /2)")
print(f"  Flatten:    [1, {128*8*8}]")
print(f"  FC1:        [1, 256]")
print(f"  Output:     [1, 10]")

## 10. Summary <a id="10-summary"></a>

| Exercise | Skill | Key Formula |
|----------|-------|---------|
| 1. CNN Block | Conv + BN + ReLU + Pool | $H_{out} = \lfloor(H + 2p - k)/s\rfloor + 1$ |
| 2. Linear Layer | MatMul + Add vs Gemm | $Y = XW + b$ |
| 3. Attributes | Inspect and modify | kernel, stride, dilation, group |
| 4. Op Analysis | Profile operator usage | Category distribution |
| 5. NumPy Verify | Correctness testing | $\|Y_{ONNX} - Y_{NumPy}\|_\infty < \epsilon$ |
| 6. Composition | GELU, Swish from primitives | $x \cdot \Phi(x)$ |
| 7. Performance | Fused vs decomposed | Gemm speedup over MatMul+Add |
| Challenge | Mini-VGG | Full CNN architecture |

### Key Takeaways

- Standard ONNX operators compose into any neural architecture
- Fused operators (Gemm, BatchNormalization) are more efficient than decomposed equivalents
- Always verify numerically: $\|f_{ONNX} - f_{reference}\|_\infty < \epsilon$
- Attributes control operator behavior without changing the graph topology